# Papers finder & screener — v5 (Stage 1: stabilised)
Run the cells in order. Secrets come from environment variables or Colab Secrets — never type keys into this notebook.

In [ ]:
# Stage 1 — install pinned, known-compatible versions.
# (pyalex 0.21 sends the OpenAlex key; langchain-openai talks to Groq's OpenAI-compatible API.)
!pip install -q "pyalex>=0.21,<1" "langchain-openai>=1.0,<2" "pydantic>=2.6,<3"

## 1. Configuration and secrets

In [ ]:
# === Configuration and secrets ===========================================
# Secrets are NEVER written in this notebook. They are read, in this order, from:
#   1. environment variables (works locally, in VS Code, in Jupyter, in CI);
#   2. Google Colab "Secrets" (the key icon in the left sidebar), if running in Colab.
# Old Colab secret names ("OpenAlex", "GROQ") are still accepted as aliases.
import os
import sys # to catch if we are running on google colab
from dotenv import load_dotenv  #to cache secrets in a .env file for local development


def get_secret(name: str, aliases: tuple[str, ...] = ()) -> str | None:
    """Return a secret value, or None if it is not configured anywhere."""
    for key in (name, *aliases):
        value = os.environ.get(key)
        if value:
            return value.strip()
    try:
        from google.colab import userdata  # only exists inside Colab
    except ImportError:
        return None
    for key in (name, *aliases):
        try:
            value = userdata.get(key)
        except Exception:  # SecretNotFoundError / NotebookAccessError
            continue
        if value:
            return value.strip()
    return None

# Automatic Google Colab ambient detection 
IN_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ

if IN_COLAB:
    print("Ambiente detetado: Google Colab")
    OPENALEX_API_KEY = get_secret("OPENALEX_API_KEY", aliases=("OpenAlex",))
    GROQ_API_KEY = get_secret("GROQ_API_KEY", aliases=("GROQ",))
    CONTACT_EMAIL = get_secret("CONTACT_EMAIL")  # optional; never hard-code a personal address
else:
    print("Ambiente detetado: Local / VS Code")
    load_dotenv()  # load secrets from .env file if it exists
    # Read the values from the environment variables (or .env file)
    OPENALEX_API_KEY = os.getenv("OPENALEX_API_KEY")
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")
    CONTACT_EMAIL = os.getenv("CONTACT_EMAIL")

GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROQ_MODEL = "openai/gpt-oss-20b"   # replacement recommended by Groq for llama-3.1-8b-instant
LLM_TEMPERATURE = 0.0               # screening is classification: keep it as deterministic as possible
PROMPT_VERSION = "relevance-v1"     # change this string whenever you edit the prompt text

print("OpenAlex key configured: ", bool(OPENALEX_API_KEY))
print("Groq key configured:     ", bool(GROQ_API_KEY))
print("Contact email configured:", bool(CONTACT_EMAIL)) # optional

## 2. Retrieval (OpenAlex)
One function, `fetch_openalex`, replaces `featcher_v1`, `fetch_papers` and `fetch_papers2`.

In [ ]:
# === Retrieval: one OpenAlex function instead of three ====================
import json
from datetime import datetime, timezone

import pyalex
from pyalex import Works

pyalex.config.api_key = OPENALEX_API_KEY        # free key; required by OpenAlex since Feb 2026
pyalex.config.email = CONTACT_EMAIL             # optional contact address (sent as a header)
pyalex.config.max_retries = 3                   # pyalex default is 0 (fail on first error)
pyalex.config.retry_backoff_factor = 0.5
pyalex.config.retry_http_codes = [429, 500, 502, 503, 504]

OPENALEX_FIELDS = [
    "id", "doi", "title", "publication_year", "authorships",
    "abstract_inverted_index", "keywords",
]
MAX_RECORDS = 10_000        # our own safety cap (pyalex's paginate() default n_max is also 10 000)
SCHEMA_VERSION = 1          # version of the record dictionaries written to JSON


def reconstruct_abstract(inverted_index) -> str:
    """Rebuild an abstract from OpenAlex's inverted index {"word": [positions]}.

    Robust to None, empty dicts, non-dict values, non-integer or negative positions,
    and gaps in the positions (gaps are skipped instead of producing double spaces).
    """
    if not isinstance(inverted_index, dict) or not inverted_index:
        return ""
    words_by_position: dict[int, str] = {}
    for word, positions in inverted_index.items():
        if not isinstance(word, str) or not isinstance(positions, (list, tuple)):
            continue
        for position in positions:
            if isinstance(position, int) and not isinstance(position, bool) and position >= 0:
                words_by_position[position] = word
    return " ".join(words_by_position[p] for p in sorted(words_by_position)).strip()


def normalise_doi(doi) -> str | None:
    """'https://doi.org/10.1/ABC' -> '10.1/abc'. DOIs are case-insensitive."""
    if not isinstance(doi, str):
        return None
    value = doi.strip().lower()
    for prefix in ("https://doi.org/", "http://doi.org/", "https://dx.doi.org/",
                   "http://dx.doi.org/", "doi:"):
        if value.startswith(prefix):
            value = value[len(prefix):]
    value = value.strip()
    return value if value.startswith("10.") else None


def work_to_record(work: dict, query: str, retrieved_at: str) -> dict:
    """Convert one OpenAlex work into our flat, consistent record (schema version 1)."""
    authors = []
    for authorship in work.get("authorships") or []:
        author = (authorship or {}).get("author") or {}
        name = author.get("display_name")
        if name:
            authors.append(name)

    keywords = []
    for entry in work.get("keywords") or []:
        name = entry.get("display_name") if isinstance(entry, dict) else None
        if name:
            keywords.append(name)

    abstract = reconstruct_abstract(work.get("abstract_inverted_index"))
    title = work.get("title")
    title = title.strip() if isinstance(title, str) and title.strip() else None
    year = work.get("publication_year")

    return {
        "schema_version": SCHEMA_VERSION,
        "source": "openalex",
        "source_id": work.get("id"),
        "doi": normalise_doi(work.get("doi")),
        "title": title,                       # None, not a fake "No Title Available" string
        "year": year if isinstance(year, int) else None,
        "authors": authors,
        "keywords": keywords,
        "abstract": abstract,                 # "" when OpenAlex has no abstract
        "has_abstract": bool(abstract),
        "query": query,
        "retrieved_at": retrieved_at,
    }


def fetch_openalex(query: str, limit: int = 100) -> tuple[list[dict], dict]:
    """Search OpenAlex and return (records, run_log). `limit` is exact: 1..MAX_RECORDS."""
    query = (query or "").strip()
    if not query:
        raise ValueError("The search query is empty.")
    if not isinstance(limit, int) or not 1 <= limit <= MAX_RECORDS:
        raise ValueError(f"limit must be an integer between 1 and {MAX_RECORDS}, got {limit!r}")

    started_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    per_page = min(limit, 200)   # OpenAlex accepts 1..200 per page
    pager = Works().search(query).select(OPENALEX_FIELDS).paginate(per_page=per_page, n_max=limit)

    records, seen_keys, duplicates, pages = [], set(), 0, 0
    for page in pager:
        pages += 1
        for work in page:
            record = work_to_record(work, query, started_at)
            key = record["source_id"] or record["doi"]
            if key and key in seen_keys:
                duplicates += 1
                continue
            if key:
                seen_keys.add(key)
            records.append(record)
            if len(records) >= limit:
                break
        print(f"  page {pages}: {len(records)} records so far")
        if len(records) >= limit:
            break

    run_log = {
        "source": "openalex",
        "query": query,
        "requested_limit": limit,
        "started_at": started_at,
        "finished_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "pages": pages,
        "records": len(records),
        "duplicates_skipped": duplicates,
        "without_abstract": sum(not r["has_abstract"] for r in records),
        "pyalex_version": getattr(pyalex, "__version__", "unknown"),
    }
    return records, run_log


def save_json(data, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

### Run a search

In [ ]:
# === Run a search (replaces the old 3-way function menu) ===================
query = input("Search query (e.g. 'human-in-the-loop machine learning'): ").strip()
limit_text = input("How many records? (1-10000, default 100): ").strip() or "100"
while not limit_text.isdigit() or not 1 <= int(limit_text) <= MAX_RECORDS:
    limit_text = input(f"Please type a whole number between 1 and {MAX_RECORDS}: ").strip()

papers, run_log = fetch_openalex(query, limit=int(limit_text))
save_json(papers, "papers_dataset.json")
save_json(run_log, "search_run.json")
print(json.dumps(run_log, indent=2))

## 3. Relevance screening (Groq)

In [ ]:
# === Relevance screening (Groq via LangChain) ==============================
import json
from datetime import datetime, timezone

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, SecretStr, StrictBool, ValidationError


class RelevanceAssessment(BaseModel):
    """The only shape we accept back from the model."""
    is_relevant: StrictBool                    # must be a real JSON true/false
    reason: str = Field(min_length=1, max_length=1000)


def extract_json_object(text: str) -> dict:
    """Return the first JSON object found in `text`.

    Handles plain JSON, ```json fenced blocks and short preambles such as
    'Here is the answer: {...}'. Raises ValueError if no object can be decoded.
    """
    if not isinstance(text, str) or not text.strip():
        raise ValueError("The model returned an empty response.")
    decoder = json.JSONDecoder()
    for start, char in enumerate(text):
        if char != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(text, start)
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict):
            return obj
    raise ValueError("No JSON object found in the model response.")


def parse_relevance_response(text: str) -> RelevanceAssessment:
    """Decode AND validate. Raises ValueError with a readable message on failure."""
    data = extract_json_object(text)
    try:
        return RelevanceAssessment.model_validate(data)
    except ValidationError as exc:
        raise ValueError(f"JSON did not match the expected schema: {exc.errors()}") from exc


prompt = PromptTemplate.from_template(
    """You are screening scientific articles for a literature review.
Research topic: {topic}

Decide whether the article below is relevant to the research topic, using ONLY
the title, abstract and keywords given. Do not use outside knowledge about the paper.

Title: {title}
Abstract: {abstract}
Keywords: {keywords}

Respond with a single JSON object and nothing else, in exactly this form:
{{"is_relevant": true or false, "reason": "one or two sentences explaining the decision"}}
"""
)

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is not set.")

llm = ChatOpenAI(
    api_key=SecretStr(GROQ_API_KEY),
    base_url=GROQ_BASE_URL,
    model=GROQ_MODEL,
    temperature=LLM_TEMPERATURE,
    timeout=60,        # seconds; never wait forever
    max_retries=2,     # retries on connection errors / 429 / 5xx
)
# Groq "JSON object mode": the response is guaranteed to be syntactically valid JSON.
# We still validate the *shape* ourselves with RelevanceAssessment.
chain = prompt | llm.bind(response_format={"type": "json_object"})


def screen_record(record: dict, topic: str) -> dict:
    """Return a NEW dict with screening fields added (the input is not modified)."""
    result = dict(record)
    result.update({
        "screening_topic": topic,
        "screening_model": GROQ_MODEL,
        "screening_prompt_version": PROMPT_VERSION,
        "screened_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "is_relevant": None,
        "reason": None,
        "screening_status": None,      # "ok" | "skipped_no_abstract" | "failed"
        "screening_error": None,
        "raw_response": None,
    })
    if not record.get("has_abstract"):
        # Deliberate: a title alone is not enough evidence. A human must look at it.
        result["screening_status"] = "skipped_no_abstract"
        result["reason"] = "No abstract available; needs manual or full-text screening."
        return result
    try:
        response = chain.invoke({
            "topic": topic,
            "title": record.get("title") or "(no title)",
            "abstract": record.get("abstract"),
            "keywords": ", ".join(record.get("keywords") or []) or "(none)",
        })
        result["raw_response"] = response.content
        assessment = parse_relevance_response(response.content)
        result["is_relevant"] = assessment.is_relevant
        result["reason"] = assessment.reason
        result["screening_status"] = "ok"
    except Exception as exc:          # network, rate limit, parse or validation error
        result["screening_status"] = "failed"
        result["screening_error"] = f"{type(exc).__name__}: {exc}"
    return result

### Run screening

In [ ]:
# === Run screening on the saved dataset ====================================
papers = load_json("papers_dataset.json")
# The topic now comes from the search you ran, not from a hard-coded string.
# You may type a more precise research topic; press Enter to reuse the query.
default_topic = papers[0]["query"] if papers and papers[0].get("query") else ""
typed = input(f"Research topic for screening [{default_topic}]: ").strip()
research_topic = typed or default_topic
if not research_topic:
    raise ValueError("No research topic given.")

print("NOTE: titles and abstracts are sent to Groq, an external service.")
evaluated = []
for i, paper in enumerate(papers, 1):
    label = (paper.get("title") or "(no title)")[:60]
    outcome = screen_record(paper, research_topic)
    evaluated.append(outcome)
    print(f"[{i}/{len(papers)}] {outcome['screening_status']:<20} {outcome['is_relevant']!s:<5} {label}")

save_json(evaluated, "evaluated_dataset.json")
counts = {}
for item in evaluated:
    counts[item["screening_status"]] = counts.get(item["screening_status"], 0) + 1
print("Summary:", counts)

## 4. Self-tests (no network needed)

In [ ]:
# === Stage 1 self-tests (no network needed) — run after cells 2, 3 and 5 ===
def run_stage1_tests():
    # reconstruct_abstract
    assert reconstruct_abstract(None) == ""
    assert reconstruct_abstract({}) == ""
    assert reconstruct_abstract("not a dict") == ""
    assert reconstruct_abstract({"The": [0, 3], "cat": [1], "sat": [2], "end": [4]}) == "The cat sat The end"
    assert reconstruct_abstract({"a": [0], "b": [5]}) == "a b"              # gap -> no double space
    assert reconstruct_abstract({"a": [0], "bad": ["x", -1, None]}) == "a"   # malformed positions ignored
    assert reconstruct_abstract({"only": []}) == ""                         # original code crashed here
    # normalise_doi
    assert normalise_doi("https://doi.org/10.1000/ABC") == "10.1000/abc"
    assert normalise_doi(None) is None and normalise_doi("") is None and normalise_doi("n/a") is None
    # work_to_record
    work = {
        "id": "https://openalex.org/W1", "doi": "https://doi.org/10.1/X", "title": "  T  ",
        "publication_year": 2024,
        "authorships": [{"author": {"display_name": "Ada"}}, {"author": None}, {}],
        "keywords": [{"display_name": "HITL"}, "junk", {"display_name": None}],
        "abstract_inverted_index": {"Hello": [0], "world": [1]},
    }
    rec = work_to_record(work, "q", "2026-01-01T00:00:00+00:00")
    assert rec["doi"] == "10.1/x" and rec["title"] == "T" and rec["authors"] == ["Ada"]
    assert rec["keywords"] == ["HITL"] and rec["abstract"] == "Hello world" and rec["has_abstract"]
    empty = work_to_record({}, "q", "t")
    assert empty["title"] is None and empty["abstract"] == "" and empty["has_abstract"] is False
    assert set(rec) == set(empty)                                           # same keys always
    # fetch_openalex argument validation
    for bad in (0, -5, MAX_RECORDS + 1, "10"):
        try:
            fetch_openalex("x", limit=bad)
            raise AssertionError(f"limit={bad!r} should be rejected")
        except ValueError:
            pass
    # response parsing
    assert parse_relevance_response('{"is_relevant": true, "reason": "ok"}').is_relevant is True
    assert parse_relevance_response('```json\n{"is_relevant": false, "reason": "no"}\n```').is_relevant is False
    assert parse_relevance_response('Answer: {"is_relevant": true, "reason": "r"} done').reason == "r"
    for bad_text in ("", "no json here", '{"is_relevant": "yes", "reason": "r"}',
                     '{"is_relevant": true}', '{"is_relevant": true, "reason": ""}'):
        try:
            parse_relevance_response(bad_text)
            raise AssertionError(f"should fail: {bad_text!r}")
        except ValueError:
            pass  
    # missing abstract is never sent to the model
    skipped = screen_record(empty, "topic")
    assert skipped["screening_status"] == "skipped_no_abstract" and skipped["is_relevant"] is None
    print("All Stage 1 tests passed.")


run_stage1_tests()